In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import cv2
import matplotlib.pyplot as plt
from skimage.color import rgb2gray
from skimage.filters import threshold_otsu

# 1. Carga una imagen de prueba de tu Google Drive
ruta_imagen = "/content/drive/MyDrive/DatasetCOLEAF/train/manganese-Mn/Mn (83).jpg"
imagen_rgb = cv2.cvtColor(cv2.imread(ruta_imagen), cv2.COLOR_BGR2RGB)

# 2. Convierte a escala de grises y calcula la magia: el umbral de Otsu
gris = rgb2gray(imagen_rgb)
umbral = threshold_otsu(gris)

# 3. Crea la silueta (mascara binaria)
# En CoLeaf el fondo es claro y la hoja oscura, por eso usamos "menor que" (<)
silueta = gris < umbral

# 4. Graficamos el antes y el después
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(imagen_rgb)
ax[0].set_title("Imagen Original")
ax[1].imshow(silueta, cmap='gray')
ax[1].set_title(f"Silueta de Otsu (Corte matemático: {umbral:.2f})")
plt.show()

In [ ]:
import os
import cv2
import numpy as np
from skimage.measure import label, regionprops
from skimage.color import rgb2gray
from skimage.filters import threshold_otsu
import matplotlib.pyplot as plt

def crop_leaf_with_regionprops(input_path, output_path):
    # 1. Leer la imagen
    image = cv2.imread(input_path)
    if image is None:
        return
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # 2. Convertir a escala de grises para encontrar el contraste
    gray = rgb2gray(image_rgb)

    # 3. Aplicar umbral (Otsu) para separar la hoja del fondo
    # Como el fondo de CoLeaf es claro y la hoja oscura, invertimos la lógica
    thresh = threshold_otsu(gray)
    binary = gray < thresh # La hoja será True (blanco), el fondo False (negro)

    # 4. Etiquetar las regiones encontradas
    label_img = label(binary)
    regions = regionprops(label_img)

    if not regions:
        # Failsafe: Si no detecta nada, guarda la original
        cv2.imwrite(output_path, image)
        return

    # 5. Encontrar la región más grande (asumimos que es la hoja, no una manchita de polvo)
    largest_region = max(regions, key=lambda r: r.area)

    # 6. Obtener las coordenadas de la caja delimitadora (Bounding Box)
    min_row, min_col, max_row, max_col = largest_region.bbox

    # 7. Recortar la imagen original usando esas coordenadas
    cropped_image = image[min_row:max_row, min_col:max_col]

    # 8. Guardar la imagen recortada
    cv2.imwrite(output_path, cropped_image)

def process_dataset(original_dir, new_dir):

    for root, dirs, files in os.walk(original_dir):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                # Construir rutas
                input_file_path = os.path.join(root, file)

                # Crear la misma estructura de carpetas en el nuevo directorio
                relative_path = os.path.relpath(root, original_dir)
                output_folder = os.path.join(new_dir, relative_path)
                os.makedirs(output_folder, exist_ok=True)

                output_file_path = os.path.join(output_folder, file)

                # Procesar y recortar
                crop_leaf_with_regionprops(input_file_path, output_file_path)
    print(f" Dataset guardado en: {new_dir}")

RUTA_DATASET_ORIGINAL = "/content/drive/MyDrive/Replicado_80_20/DatasetCOLEAF_80_20_dividido"
RUTA_DATASET_RECORTADO = "/content/si/DatasetCOLEAF_Recortado"



In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126

!pip install timm

!pip install pytorch-lightning torchview imageio matplotlib scikit-learn pandas seaborn graphviz transformers scikit-image

In [ ]:
!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cpu

In [ ]:
#Cuando se corren los VIT
# 2. Instalación desde el repositorio de TIMM (para modelos actualizados)
!pip install git+https://github.com/rwightman/pytorch-image-models.git

# 3. Resto de librerías del ecosistema ML y gráficas
!pip install pytorch-lightning torchview imageio matplotlib scikit-learn pandas seaborn graphviz transformers scikit-image

Usos **MAMBA**

In [ ]:
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

In [ ]:
!pip install ninja packaging
!pip install causal-conv1d==1.4.0

In [ ]:
# 1. Instalaciones críticas de Mamba
!pip install mamba-ssm --no-build-isolation

# 2. Instalación desde el repositorio de TIMM (para modelos actualizados)
!pip install git+https://github.com/rwightman/pytorch-image-models.git

# 3. Resto de librerías del ecosistema ML y gráficas
!pip install pytorch-lightning torchview imageio matplotlib scikit-learn pandas seaborn graphviz transformers scikit-image

In [ ]:
# --- 1. LIBRERÍAS ESTÁNDAR ---
import os
import sys
import gc
import json
import shutil
import warnings
import requests
import multiprocessing
from glob import glob
from datetime import datetime

warnings.filterwarnings("ignore")

# --- 2. MANEJO DE DATOS Y VISUALIZACIÓN ---
import numpy as np
import pandas as pd
import seaborn as sn
import matplotlib.pyplot as plt
import imageio
import graphviz
from PIL import Image
from tqdm.notebook import tqdm

# --- 3. MÉTRICAS Y MACHINE LEARNING TRADICIONAL ---
from sklearn import metrics as sk_metrics
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             f1_score, classification_report, accuracy_score)
from skimage.measure import label, regionprops

# --- 4. PYTORCH CORE ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.utils.data.sampler import SubsetRandomSampler
from torch.cuda.amp import GradScaler

# --- 5. TORCHVISION ---
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import torchvision.models as models
from torchvision import datasets
from torchvision.datasets import ImageFolder
from torchvision.transforms import ToTensor

# --- 6. PYTORCH LIGHTNING Y ECOSISTEMA ---
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import Callback, EarlyStopping, ModelCheckpoint
from torchmetrics.classification import Accuracy
from torchview import draw_graph
import timm

# --- 7. HUGGING FACE TRANSFORMERS ---
from transformers import (
    ViTImageProcessor,
    ViTForImageClassification,
    BeitImageProcessor,
    BeitForImageClassification
)

# --- VERIFICACIÓN DE ENTORNO COLAB ---
print(f"PyTorch Version: {torch.__version__}")
print(f"GPU Disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Asignada: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
import matplotlib.pyplot as plt

def Show_data(DATA_PATH, type):
    # Asegúrate de usar la variable correcta para evitar sobreescribir el DATA_PATH global
    full_path = os.path.join(DATA_PATH, type)
    list_dire_clases= [f.path for f in os.scandir(full_path) if f.is_dir()]
    list_name_clases = [os.path.basename(x) for x in list_dire_clases]

    cantidades = []

    for class_path, class_name in zip(list_dire_clases, list_name_clases):
        # 1. Obtenemos todos los archivos de la carpeta
        todos_los_archivos = os.listdir(class_path)

        # 2. Filtramos solo los que sean imágenes sin importar si es mayúscula/minúscula
        extensiones_validas = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
        files_annot = [f for f in todos_los_archivos if f.lower().endswith(extensiones_validas)]

        print(f"Clase {class_name} tiene {len(files_annot)} registros")
        cantidades.append(len(files_annot))

    plt.bar(list_name_clases, cantidades)
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
DATA_PATH="/content/drive/MyDrive/DatasetCOLEAF_Recortado"
Show_data(DATA_PATH, "train")
Show_data(DATA_PATH, "valid")
Show_data(DATA_PATH, "test")

# Preprocesamiento


In [ ]:
train_dir = '/content/drive/MyDrive/Replicado_80_20/DatasetCOLEAF_80_20_dividido/train'
valid_dir = '/content/drive/MyDrive/Replicado_80_20/DatasetCOLEAF_80_20_dividido/valid'
#test_dir = '/content/drive/MyDrive/DatasetCOLEAF_Recortado/test'

In [ ]:
base_dir   = "/content/drive/MyDrive/Replicado_80_20/DatasetCOLEAF_80_20_dividido"
data_dir_train = '/content/drive/MyDrive/Replicado_80_20/DatasetCOLEAF_80_20_dividido/train'
data_dir_valid = '/content/drive/MyDrive/Replicado_80_20/DatasetCOLEAF_80_20_dividido/valid'
#data_dir_test = '/content/drive/MyDrive/DatasetCOLEAF_Recortado/test'

In [ ]:
# Usaremos ImageFolder de torchvision, que es más adecuado
# para datos organizados por carpetas.
# Mantenemos el nombre MyDataset por compatibilidad con DataModule.
class MyDataset(datasets.ImageFolder):
    def __init__(self, root, transform=None):
        # ImageFolder automáticamente encuentra las clases a partir
        # de los nombres de las carpetas dentro de 'root'
        super().__init__(root, transform=transform)

In [ ]:
class DataModule(pl.LightningDataModule):
    def __init__(self, dat_norm, img_size=224, batch_size=64,
                 train_dir=None, valid_dir=None, test_dir=None, categories=None):

        super().__init__()
        self.img_size = img_size

        # Transformaciones (las mismas)
        self.transform = transforms.Compose([
            transforms.Resize(size=(self.img_size, self.img_size)),
            transforms.ToTensor(),
            transforms.Normalize(dat_norm[0], dat_norm[1]),
        ])
        self.train_transform = transforms.Compose([
            # 1. Ajuste base
            transforms.Resize(size=(self.img_size, self.img_size)),

            # 2. Rotaciones y volteos (El celular de cabeza o de lado)
            transforms.RandomRotation(degrees=45),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),

            # 3. Encuadres imperfectos en el campo
            # translate=(0.1, 0.1) mueve la imagen hasta un 10% en cualquier dirección
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            # distortion_scale=0.2 inclina la foto ligeramente en 3D (30% de probabilidad)
            transforms.RandomPerspective(distortion_scale=0.2, p=0.3),

            # 4. Problemas de iluminación (Sol fuerte, nublado, sombras)
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),

            # 5. Problemas de enfoque / Temblor de mano
            # Se aplica un desenfoque gaussiano solo al 30% de las fotos (p=0.3)
            transforms.RandomApply([transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))], p=0.3),

            # 6. Conversión a tensores
            transforms.ToTensor(),
            transforms.Normalize(dat_norm[0], dat_norm[1]),
        ])

        self.batch_size = batch_size
        self.num_workers = multiprocessing.cpu_count()

        # Guardamos las rutas para cargarlas en setup()
        self.train_dir = train_dir
        self.valid_dir = valid_dir
        self.test_dir = test_dir

        # Estos se inicializarán como None y se cargarán en setup()
        self.traindata = None
        self.validdata = None
        self.testdata = None

    # setup() para cargar los datos usando ImageFolder (MyDataset)
    def setup(self, stage=None):
        if stage == 'fit' or stage is None:
            if self.train_dir:
                # La ruta del directorio de entrenamiento se pasa a MyDataset (ImageFolder)
                self.traindata = MyDataset(self.train_dir, self.train_transform)
            if self.valid_dir:
                self.validdata = MyDataset(self.valid_dir, self.transform)

        if stage == 'test' or stage is None:
            if self.test_dir:
                self.testdata = MyDataset(self.test_dir, self.transform)

    def train_dataloader(self):
        if self.traindata is None:
            raise RuntimeError("Train data not loaded. Check your train_dir path.")
        # --- AQUÍ OCURRE LA MAGIA DEL BALANCEO ---
        # 1. Contamos cuántas imágenes hay de cada clase
        targets = self.traindata.targets
        class_counts = torch.bincount(torch.tensor(targets))

        # 2. Le damos más "peso" (importancia) a las clases que tienen menos imágenes
        class_weights = 1.0 / class_counts.float()

        # 3. Asignamos ese peso a cada imagen individual en el dataset
        sample_weights = class_weights[targets]

        # 4. Creamos el sampler. Él se encargará de pedir más veces las fotos raras
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True # Permite repetir la misma imagen rara en un mismo epoch (¡con nueva transformación!)
        )

        # OJO: Cuando usas 'sampler', NO puedes usar 'shuffle=True'
        return DataLoader(
            self.traindata,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            sampler=sampler, # Agregamos el sampler
            drop_last=True
        )

    def val_dataloader(self):
        if self.validdata is None:
             raise RuntimeError("Validation data not loaded. Check your valid_dir path.")
        return torch.utils.data.DataLoader(
            self.validdata, batch_size=self.batch_size, num_workers=self.num_workers, drop_last=False)

    def test_dataloader(self):
        if self.testdata is None:
             raise RuntimeError("Test data not loaded. Check your test_dir path.")
        return torch.utils.data.DataLoader(
            self.testdata, batch_size=self.batch_size, num_workers=0, drop_last=False)

**Modo Replicar sin test**

In [ ]:
class DataModule(pl.LightningDataModule):
    def __init__(self, dat_norm, img_size=224, batch_size=64,
                 train_dir=None, valid_dir=None, categories=None): # Se eliminó test_dir

        super().__init__()
        self.img_size = img_size

        # Transformaciones (las mismas)
        self.transform = transforms.Compose([
            transforms.Resize(size=(self.img_size, self.img_size)),
            transforms.ToTensor(),
            transforms.Normalize(dat_norm[0], dat_norm[1]),
        ])

        self.train_transform = transforms.Compose([
        # 1. Regresamos al Resize clásico. No queremos recortar ni perder bordes secos.
        transforms.Resize(size=(self.img_size, self.img_size)),

        # 2. Geometría Segura (No destruye píxeles, solo cambia la perspectiva)
        transforms.RandomRotation(degrees=45),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),

        # 3. Color Sutil (Como lo tenías en 0.1, pero bloqueando el cambio de Tono/Hue)
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0),

        # 4. Desenfoque al mínimo (Ayuda a generalizar sin borrar manchas, p=0.1)
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.1),

        # 5. Conversión y Normalización
        transforms.ToTensor(),
        transforms.Normalize(dat_norm[0], dat_norm[1]),

        # ELIMINADOS: RandomErasing y RandomAffine. Estaban tapando/moviendo las enfermedades.
        ])

        self.batch_size = batch_size
        self.num_workers = multiprocessing.cpu_count()

        # Guardamos las rutas para cargarlas en setup()
        self.train_dir = train_dir
        self.valid_dir = valid_dir
        # Se eliminó self.test_dir

        # Estos se inicializarán como None y se cargarán en setup()
        self.traindata = None
        self.validdata = None
        # Se eliminó self.testdata

    # setup() para cargar los datos usando ImageFolder (MyDataset)
    def setup(self, stage=None):
        # Ahora solo nos interesa la fase de ajuste ('fit')
        if stage == 'fit' or stage is None:
            if self.train_dir:
                # La ruta del directorio de entrenamiento se pasa a MyDataset (ImageFolder)
                self.traindata = MyDataset(self.train_dir, self.train_transform)
            if self.valid_dir:
                self.validdata = MyDataset(self.valid_dir, self.transform)

        # Se eliminó por completo el bloque "if stage == 'test': ..."

    def train_dataloader(self):
        if self.traindata is None:
            raise RuntimeError("Train data not loaded. Check your train_dir path.")

        return DataLoader(
            self.traindata,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            shuffle=True,
            drop_last=True
        )

    def val_dataloader(self):
        if self.validdata is None:
             raise RuntimeError("Validation data not loaded. Check your valid_dir path.")
        return torch.utils.data.DataLoader(
            self.validdata,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            drop_last=False
        )

    # Se eliminó por completo def test_dataloader(self)

In [ ]:
#Beta:
self.train_transform = transforms.Compose([
    # 1. Regresamos al Resize clásico. No queremos recortar ni perder bordes secos.
    transforms.Resize(size=(self.img_size, self.img_size)),

    # 2. Geometría Segura (No destruye píxeles, solo cambia la perspectiva)
    transforms.RandomRotation(degrees=45),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),

    # 3. Color Sutil (Como lo tenías en 0.1, pero bloqueando el cambio de Tono/Hue)
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0),

    # 4. Desenfoque al mínimo (Ayuda a generalizar sin borrar manchas, p=0.1)
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.1),

    # 5. Conversión y Normalización
    transforms.ToTensor(),
    transforms.Normalize(dat_norm[0], dat_norm[1]),

    # 8. Normalización (Siempre al final)
    transforms.Normalize(dat_norm[0], dat_norm[1]),
])

# Train and Test

In [ ]:
#2
scaler = GradScaler()

def train(model, criterion, device, train_loader, valid_loader, optimizer, scheduler, num_epochs, porc=0):
    log_interval = 10
    print("Begin training...")
    total = (len(train_loader) + len(valid_loader))*num_epochs
    n = len(list(model.parameters()))

    if porc > 0:
        print(f"APLICANDO FINE TUNING. \n Descongelando el {porc*100}% de capas.")
        for i, param in enumerate(model.parameters()):
            if i > n*(1-porc):
                param.requires_grad = True
    else:
        print(f"APLICANDO TRANSFER LEARNING. \n Entrenando Bloque añadido.")

    history = []
    lr_history = []
    best_f1 = 0
    best_accuracy = 0

    for epoch in range(1, num_epochs+1):
        model.train()
        running_train_loss = 0.0
        all_preds = []
        all_labels = []

        for batch_idx, (img, target) in enumerate(train_loader):
            img, target = img.to(device, non_blocking=True), target.to(device, non_blocking=True)

            # Limpiar gradientes del lote anterior
            optimizer.zero_grad()

            # Forward and backward propagation with mixed precision
            with torch.amp.autocast('cuda'):
                output = model(img)
                train_loss = criterion(output.float(), target)
            # Scales the loss, and calls backward() to create scaled gradients
            scaler.scale(train_loss).backward()

            # Unscales the gradients of optimizer's assigned params in-place
            scaler.step(optimizer)
            scaler.update()

            # Show progress
            running_train_loss += train_loss.item()
            all_preds.extend(output.float().argmax(dim=1).cpu().numpy())
            all_labels.extend(target.cpu().numpy())

            if batch_idx % log_interval == 0:
                print(f'Train Epoch: {epoch} [{batch_idx * len(img)}/{len(train_loader.dataset)} ({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {train_loss.item():.3f}')

        # Métricas de Entrenamiento de la época completa
        train_loss_value = running_train_loss / len(train_loader)
        train_f1 = f1_score(all_labels, all_preds, average='macro')

        # Validation Loop
        val_loss_value, accuracy, val_f1 = validation(model, valid_loader)
        history.append((train_loss_value, val_loss_value, accuracy, val_f1))

        # Step the scheduler
        scheduler.step(val_f1)
        lr_history.append(optimizer.param_groups[0]['lr'])

        print(f'COMPLETED Train Epoch {epoch}: Training Loss is: {train_loss_value:.3f} - Validation Loss is: {val_loss_value:.3f} - Accuracy is {accuracy:.2f}% - F1 Score is {val_f1:.4f}')

        if val_f1 > best_f1:
            print(f'Validation F1-Score increment: {best_f1:.4f} ⮕ {val_f1:.4f} | Acc: {best_accuracy:.2f}% ⮕ {accuracy:.2f}% \t Saving The Model')
            saveModel(model)
            best_accuracy = accuracy
            best_f1 = val_f1

    return model, history, lr_history

In [ ]:
#2
def validation(model, valid_loader):
    model.eval()
    running_vall_loss = 0.0
    total = 0.0
    running_accuracy = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for img, target in valid_loader:
            img, target = img.to(device), target.to(device)
            predicted_outputs = model(img)
            val_loss = criterion(predicted_outputs, target)
            _, predicted = torch.max(predicted_outputs, 1)
            running_vall_loss += val_loss.item()
            total += target.size(0)
            running_accuracy += (predicted == target).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(target.cpu().numpy())

        val_loss_value = running_vall_loss / len(valid_loader)
        accuracy = (100 * running_accuracy / total)
        val_f1 = f1_score(all_labels, all_preds, average='macro')

    return val_loss_value, accuracy, val_f1

In [ ]:
#2
from sklearn.metrics import f1_score, classification_report
import torch
def test(model, criterion, device, test_loader, class_names):
    model.eval()
    test_loss = 0
    all_preds = []
    all_labels = []

    print("Iniciando Test...")
    with torch.no_grad():
        for img, target in test_loader:
            img, target = img.to(device), target.to(device)
            output = model(img)
            test_loss += criterion(output, target).item()
            # 2. Obtenemos la clase predicha directamente
            _, predicted = torch.max(output, 1)

            # 3. Pasamos a CPU y guardamos en la lista (ESTO ES LO EFICIENTE)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(target.cpu().numpy())

    test_loss /= len(test_loader)
    y_true = all_labels
    y_pred = all_preds
    print(f'Test set: Average loss: {test_loss:.4f}')
    print(f'Test set: Accuracy score: {sk_metrics.accuracy_score(y_true, y_pred):.4f}')
    print(f'Test set: Macro F1 score: {sk_metrics.f1_score(y_true, y_pred, average="macro"):.4f}')
      # Imprimir el reporte detallado por clase

    print("\nREPORTE DETALLADO POR CLASE:")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
    print('='*60 + '\n')
    return y_pred, y_true

## **Modelo VGG-16**

In [ ]:
def saveModel(model):
    dir_Save ="/content/new_dataVGG16"
    os.makedirs(dir_Save, exist_ok=True)
    path = os.path.join(dir_Save, "NetModel.pth")
    torch.save(model.state_dict(), path)

In [ ]:
class CNN_VGG16(nn.Module):
    def __init__(self, num_classes = 10, pretrained = True):
        super().__init__()

        self.num_classes = num_classes

        # Para VGG16
        self.vgg = timm.create_model("vgg16", pretrained = pretrained)
        if pretrained:
          # freeze  weights
            for param in self.vgg.parameters():
              param.requires_grad = False

        self.numfeat = self.vgg.get_classifier().in_features

        block = nn.Sequential(
            nn.Linear(self.numfeat, 1024),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.Dropout(0.4),
            nn.Linear(512, self.num_classes))

        self.vgg.head.fc = block

    def forward(self, x):
        out = self.vgg(x)
        return out

    def get_mean_std(self):
      return self.vgg.default_cfg["mean"], self.vgg.default_cfg["std"]

### **Configurando Hiperparámetros**

In [ ]:
import gc
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(47)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
num_class  = 9
lr         = 1e-4
num_epochs  = 100
batch_size = 64
img_size   = 224

model_cnn = CNN_VGG16(num_classes=num_class).to(device)
#criterion  = nn.CrossEntropyLoss().to(device)
optimizer =  torch.optim.AdamW(model_cnn.parameters(), lr=lr)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)

In [ ]:
dm = DataModule(model_cnn.get_mean_std(), img_size = img_size, batch_size=batch_size, train_dir = train_dir, valid_dir=valid_dir)
dm.setup()
train_loader = dm.train_dataloader()
valid_loader = dm.val_dataloader()
#test_loader = dm.test_dataloader()

In [ ]:
# =================================================================
# CÁLCULO DE PESOS
# =================================================================
# 1. Extraemos las etiquetas usando tu variable 'dm'
etiquetas_train = dm.traindata.targets

# 2. Contamos cuántas imágenes hay por clase y aplicamos la fórmula
conteo_clases = torch.bincount(torch.tensor(etiquetas_train))
total_muestras = len(etiquetas_train)
num_clases = len(conteo_clases)

pesos_clases = total_muestras / (num_clases * conteo_clases.float())
pesos_tensor = pesos_clases.to(device).float()


print("="*50)
print(f"⚖️ Pesos matemáticos calculados para las {num_clases} clases:")
print(pesos_tensor)
print("="*50)

# 3 CRITERION CON PESOS INYECTADOS
#criterion = nn.CrossEntropyLoss(weight=pesos_tensor, label_smoothing=0.1).to(device)
criterion = nn.CrossEntropyLoss(weight=pesos_tensor).to(device)

In [ ]:
model_cnn, history, lr_history = train(model_cnn, criterion, device, train_loader, valid_loader, optimizer, scheduler, num_epochs, porc = 0.3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Simulación de Datos (Reemplaza 'history' con tus datos reales) ---
# history = [ [loss_train, loss_valid, acc_train, acc_valid], ... ]
arr = np.array(history)
num_epochs = arr.shape[0]
epochs_range = np.arange(0, num_epochs) # Creamos el rango de 0 a N
# ----------------------------------------------------------------------

# Definimos cada cuánto queremos ver los números en el eje X para que no se amontonen
# Si son pocas épocas (menos de 20), mostramos todas. Si son muchas, saltamos.
step = max(1, num_epochs // 10)
mis_ticks = np.arange(0, num_epochs, step)

# ==========================================
# GRÁFICO 1: PÉRDIDA (LOSS)
# ==========================================
plt.figure(figsize=(10, 6))
plt.title('Historial de Pérdida (Loss)', fontsize=16)

plt.plot(epochs_range, arr[:, 0], label='Loss_train', linewidth=2.5)
plt.plot(epochs_range, arr[:, 1], label='Loss_valid', linewidth=2.5, linestyle='--')

plt.ylabel('Pérdida (Loss)', fontsize=12)
plt.xlabel('Época (Epoch)', fontsize=12)

# AQUÍ ASEGURAMOS LOS NUMERITOS EN EL EJE X
plt.xticks(mis_ticks)

plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


# ==========================================
# GRÁFICO 2: PRECISIÓN (ACCURACY) - Sin Acc_valid
# ==========================================
plt.figure(figsize=(10, 6))
plt.title('Historial de Precisión (Accuracy)', fontsize=16)

# Solo ploteamos la columna 2 (Train), ignoramos la 3 (Valid)
plt.plot(epochs_range, arr[:, 2], label='Acc_train', color='green', linewidth=2.5)

plt.ylabel('Precisión (Acc)', fontsize=12)
plt.xlabel('Época (Epoch)', fontsize=12)

# AQUÍ ASEGURAMOS LOS NUMERITOS EN EL EJE X TAMBIÉN
plt.xticks(mis_ticks)

plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### **Entrenando el Modelo** - Cargando mejor Modelo


In [ ]:
model_sv = CNN_VGG16(num_classes=num_class).to(device)
path = "/content/new_dataVGG16/NetModel.pth"
model_sv.load_state_dict(torch.load(path))

In [ ]:
loader = valid_loader
labels = list(dm.validdata.class_to_idx.keys())
y_pred, y_true = test(model_sv, criterion, device, loader, labels)

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
tot = np.sum(cm, axis=1)
cm_porc = np.divide(cm, np.reshape(tot, (-1,1)))

**Pruebas**

In [ ]:
#@title Matriz de confusión en porcentajes
labels =list(dm.validdata.class_to_idx.keys())
# Calculate confusion matrix
#confusion_matrix = sk_metrics.confusion_matrix(y_true, y_pred)
df_confusion_matrix = pd.DataFrame(cm_porc, index=labels, columns=labels)

# Show confusion matrix
plt.figure(figsize=(6, 6))
sn.heatmap(df_confusion_matrix, annot=True, cbar=False, cmap='Oranges', linewidths=1, linecolor='black', fmt=".2%")
plt.xlabel('Predicted labels', fontsize=15)
plt.xticks(fontsize=12)
plt.ylabel('True labels', fontsize=15)
plt.yticks(fontsize=12);

In [ ]:
report = classification_report(y_true, y_pred, target_names=labels, output_dict=True)
print(classification_report(y_true, y_pred, target_names=labels))

# InceptionV3

In [ ]:
def saveModel(model):
    dir_Save ="/content/new_dataInceptionV3"
    os.makedirs(dir_Save, exist_ok=True)
    path = os.path.join(dir_Save, "NetModel.pth")
    torch.save(model.state_dict(), path)

Modelo InceptionV3

In [ ]:
class CNN_IV3(nn.Module):
    def __init__(self, num_classes = 10, pretrained = True):
        super().__init__()

        self.num_classes = num_classes

        # Para IncpetionV3
        self.inception_v3 = timm.create_model("inception_v3", pretrained = pretrained)
        if pretrained:
          # freeze  weights
            for param in self.inception_v3.parameters():
              param.requires_grad = False

        self.numfeat = self.inception_v3.get_classifier().in_features

        block = nn.Sequential(
            nn.Linear(self.numfeat, 1024),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.Dropout(0.4),
            nn.Linear(512, self.num_classes))

        self.inception_v3.fc = block

    def forward(self, x):
        out = self.inception_v3.forward(x)
        return out

    def get_mean_std(self):
      return self.inception_v3.default_cfg["mean"], self.inception_v3.default_cfg["std"]

In [ ]:
import gc
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(47)

gc.collect()
torch.cuda.empty_cache()

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
num_class  = 9
lr         = 1e-4
num_epochs  = 100
batch_size = 64
img_size   = 224

model_cnn = CNN_IV3(num_classes=num_class).to(device)
#criterion  = nn.CrossEntropyLoss().to(device)
optimizer =  torch.optim.AdamW(model_cnn.parameters(), lr=lr)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)

In [ ]:
dm = DataModule(model_cnn.get_mean_std(), img_size = img_size, batch_size=batch_size, train_dir = train_dir, valid_dir=valid_dir)
dm.setup()
train_loader = dm.train_dataloader()
valid_loader = dm.val_dataloader()
#test_loader = dm.test_dataloader()

In [ ]:
# =================================================================
# CÁLCULO DE PESOS
# =================================================================
# 1. Extraemos las etiquetas usando tu variable 'dm'
etiquetas_train = dm.traindata.targets

# 2. Contamos cuántas imágenes hay por clase y aplicamos la fórmula
conteo_clases = torch.bincount(torch.tensor(etiquetas_train))
total_muestras = len(etiquetas_train)
num_clases = len(conteo_clases)

pesos_clases = total_muestras / (num_clases * conteo_clases.float())
pesos_tensor = pesos_clases.to(device).float()


print("="*50)
print(f"⚖️ Pesos matemáticos calculados para las {num_clases} clases:")
print(pesos_tensor)
print("="*50)

# 3 CRITERION CON PESOS INYECTADOS
#criterion = nn.CrossEntropyLoss(weight=pesos_tensor, label_smoothing=0.1).to(device)
criterion = nn.CrossEntropyLoss(weight=pesos_tensor).to(device)

### **Entrenando el Modelo**

In [ ]:
model_cnn, history, lr_history = train(model_cnn, criterion, device, train_loader, valid_loader, optimizer, scheduler, num_epochs, porc = 0.3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Simulación de Datos (Reemplaza 'history' con tus datos reales) ---
# history = [ [loss_train, loss_valid, acc_train, acc_valid], ... ]
arr = np.array(history)
num_epochs = arr.shape[0]
epochs_range = np.arange(0, num_epochs) # Creamos el rango de 0 a N
# ----------------------------------------------------------------------

# Definimos cada cuánto queremos ver los números en el eje X para que no se amontonen
# Si son pocas épocas (menos de 20), mostramos todas. Si son muchas, saltamos.
step = max(1, num_epochs // 10)
mis_ticks = np.arange(0, num_epochs, step)

# ==========================================
# GRÁFICO 1: PÉRDIDA (LOSS)
# ==========================================
plt.figure(figsize=(10, 6))
plt.title('Historial de Pérdida (Loss)', fontsize=16)

plt.plot(epochs_range, arr[:, 0], label='Loss_train', linewidth=2.5)
plt.plot(epochs_range, arr[:, 1], label='Loss_valid', linewidth=2.5, linestyle='--')

plt.ylabel('Pérdida (Loss)', fontsize=12)
plt.xlabel('Época (Epoch)', fontsize=12)

# AQUÍ ASEGURAMOS LOS NUMERITOS EN EL EJE X
plt.xticks(mis_ticks)

plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


# ==========================================
# GRÁFICO 2: PRECISIÓN (ACCURACY) - Sin Acc_valid
# ==========================================
plt.figure(figsize=(10, 6))
plt.title('Historial de Precisión (Accuracy)', fontsize=16)

# Solo ploteamos la columna 2 (Train), ignoramos la 3 (Valid)
plt.plot(epochs_range, arr[:, 2], label='Acc_train', color='green', linewidth=2.5)

plt.ylabel('Precisión (Acc)', fontsize=12)
plt.xlabel('Época (Epoch)', fontsize=12)

# AQUÍ ASEGURAMOS LOS NUMERITOS EN EL EJE X TAMBIÉN
plt.xticks(mis_ticks)

plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### **Entrenando el Modelo** - Cargando el mejor Modelo

In [ ]:
model_sv = CNN_IV3(num_classes=num_class).to(device)
path = "/content/new_dataInceptionV3/NetModel.pth"
model_sv.load_state_dict(torch.load(path))

In [ ]:
loader = valid_loader
labels = list(dm.validdata.class_to_idx.keys())
y_pred, y_true = test(model_sv, criterion, device, loader, labels)

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
tot = np.sum(cm, axis=1)
cm_porc = np.divide(cm, np.reshape(tot, (-1,1)))

**Pruebas**

In [ ]:
#@title Matriz de confusión en porcentajes
labels =list(dm.validdata.class_to_idx.keys())
# Calculate confusion matrix
#confusion_matrix = sk_metrics.confusion_matrix(y_true, y_pred)
df_confusion_matrix = pd.DataFrame(cm_porc, index=labels, columns=labels)

# Show confusion matrix
plt.figure(figsize=(6, 6))
sn.heatmap(df_confusion_matrix, annot=True, cbar=False, cmap='Oranges', linewidths=1, linecolor='black', fmt=".2%")
plt.xlabel('Predicted labels', fontsize=15)
plt.xticks(fontsize=12)
plt.ylabel('True labels', fontsize=15)
plt.yticks(fontsize=12);

In [ ]:
report = classification_report(y_true, y_pred, target_names=labels, output_dict=True)
print(classification_report(y_true, y_pred, target_names=labels))

# ResNet200d

In [ ]:
def saveModel(model):
    dir_Save ="/content/new_data"
    os.makedirs(dir_Save, exist_ok=True)
    path = os.path.join(dir_Save, "NetModel.pth")
    torch.save(model.state_dict(), path)

### **Modelo Resnet200d**

In [ ]:
class CNN_RESNET(nn.Module):
    def __init__(self, num_classes = 4, pretrained = True):
        super().__init__()

        self.num_classes = num_classes

        # Para RESNET
        self.resnet = timm.create_model("resnet200d", pretrained = pretrained)
        if pretrained:
          # freeze  weights
            for param in self.resnet.parameters():
              param.requires_grad = False

        self.numfeat = self.resnet.get_classifier().in_features

        block = nn.Sequential(
            nn.Linear(self.numfeat, 1024),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.Dropout(0.4),
            nn.Linear(512, self.num_classes))

        self.resnet.fc = block

    def forward(self, x):
        out = self.resnet.forward(x)
        return out

    def get_mean_std(self):
      return self.resnet.default_cfg["mean"], self.resnet.default_cfg["std"]

In [ ]:
import gc
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(47)


In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
num_class  = 9
lr         = 1e-4
num_epochs  = 100
batch_size = 64
img_size   = 224

model_cnn = CNN_RESNET(num_classes=num_class).to(device)
#criterion  = nn.CrossEntropyLoss().to(device)
optimizer =  torch.optim.AdamW(model_cnn.parameters(), lr=lr)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)

In [ ]:
dm = DataModule(model_cnn.get_mean_std(), img_size = img_size, batch_size=batch_size, train_dir = train_dir, valid_dir=valid_dir)
dm.setup()
train_loader = dm.train_dataloader()
valid_loader = dm.val_dataloader()
#test_loader = dm.test_dataloader()

In [ ]:
# =================================================================
# CÁLCULO DE PESOS
# =================================================================
# 1. Extraemos las etiquetas usando tu variable 'dm'
etiquetas_train = dm.traindata.targets

# 2. Contamos cuántas imágenes hay por clase y aplicamos la fórmula
conteo_clases = torch.bincount(torch.tensor(etiquetas_train))
total_muestras = len(etiquetas_train)
num_clases = len(conteo_clases)

pesos_clases = total_muestras / (num_clases * conteo_clases.float())
pesos_tensor = pesos_clases.to(device).float()


print("="*50)
print(f"⚖️ Pesos matemáticos calculados para las {num_clases} clases:")
print(pesos_tensor)
print("="*50)

# 3 CRITERION CON PESOS INYECTADOS
#criterion = nn.CrossEntropyLoss(weight=pesos_tensor, label_smoothing=0.1).to(device)
criterion = nn.CrossEntropyLoss(weight=pesos_tensor).to(device)

***Entrenamiento del modelo***

In [ ]:
model_cnn, history, lr_history = train(model_cnn, criterion, device, train_loader, valid_loader, optimizer, scheduler, num_epochs, porc = 0.3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Simulación de Datos (Reemplaza 'history' con tus datos reales) ---
# history = [ [loss_train, loss_valid, acc_train, acc_valid], ... ]
arr = np.array(history)
num_epochs = arr.shape[0]
epochs_range = np.arange(0, num_epochs) # Creamos el rango de 0 a N
# ----------------------------------------------------------------------

# Definimos cada cuánto queremos ver los números en el eje X para que no se amontonen
# Si son pocas épocas (menos de 20), mostramos todas. Si son muchas, saltamos.
step = max(1, num_epochs // 10)
mis_ticks = np.arange(0, num_epochs, step)

# ==========================================
# GRÁFICO 1: PÉRDIDA (LOSS)
# ==========================================
plt.figure(figsize=(10, 6))
plt.title('Historial de Pérdida (Loss)', fontsize=16)

plt.plot(epochs_range, arr[:, 0], label='Loss_train', linewidth=2.5)
plt.plot(epochs_range, arr[:, 1], label='Loss_valid', linewidth=2.5, linestyle='--')

plt.ylabel('Pérdida (Loss)', fontsize=12)
plt.xlabel('Época (Epoch)', fontsize=12)

# AQUÍ ASEGURAMOS LOS NUMERITOS EN EL EJE X
plt.xticks(mis_ticks)

plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


# ==========================================
# GRÁFICO 2: PRECISIÓN (ACCURACY) - Sin Acc_valid
# ==========================================
plt.figure(figsize=(10, 6))
plt.title('Historial de Precisión (Accuracy)', fontsize=16)

# Solo ploteamos la columna 2 (Train), ignoramos la 3 (Valid)
plt.plot(epochs_range, arr[:, 2], label='Acc_train', color='green', linewidth=2.5)

plt.ylabel('Precisión (Acc)', fontsize=12)
plt.xlabel('Época (Epoch)', fontsize=12)

# AQUÍ ASEGURAMOS LOS NUMERITOS EN EL EJE X TAMBIÉN
plt.xticks(mis_ticks)

plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### **Entrenando el Modelo** - Cargando el Mejor Modelo

In [ ]:
model_sv = CNN_RESNET(num_classes=num_class).to(device)
path = "/content/new_data/NetModel.pth"
model_sv.load_state_dict(torch.load(path))

In [ ]:
loader = valid_loader
labels = list(dm.validdata.class_to_idx.keys())
y_pred, y_true = test(model_sv, criterion, device, loader, labels)

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
tot = np.sum(cm, axis=1)
cm_porc = np.divide(cm, np.reshape(tot, (-1,1)))

In [ ]:
#@title Matriz de confusión en porcentajes
labels =list(dm.validdata.class_to_idx.keys())
# Calculate confusion matrix
#confusion_matrix = sk_metrics.confusion_matrix(y_true, y_pred)
df_confusion_matrix = pd.DataFrame(cm_porc, index=labels, columns=labels)

# Show confusion matrix
plt.figure(figsize=(6, 6))
sn.heatmap(df_confusion_matrix, annot=True, cbar=False, cmap='Oranges', linewidths=1, linecolor='black', fmt=".2%")
plt.xlabel('Predicted labels', fontsize=15)
plt.xticks(fontsize=12)
plt.ylabel('True labels', fontsize=15)
plt.yticks(fontsize=12);

In [ ]:
report = classification_report(y_true, y_pred, target_names=labels, output_dict=True)
print(classification_report(y_true, y_pred, target_names=labels))

# Google Vit


In [ ]:
from transformers import ViTImageProcessor, ViTForImageClassification
import requests

In [ ]:
def saveModel(model):
    dir_Save ="/content/new_dataGVIT"
    os.makedirs(dir_Save, exist_ok=True)
    path = os.path.join(dir_Save, "NetModel.pth")
    torch.save(model.state_dict(), path)

In [ ]:
#@title google/vit-base-patch16-224
class GVIT(nn.Module):
    def __init__(self, num_classes = 10, pretrained=True):
        super().__init__()

        self.num_classes = num_classes

        ########
        processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
        self.backbone =  ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')

        if pretrained:
          # freeze  weights
            for param in self.backbone.parameters():
              param.requires_grad = False

        self.numfeat = self.backbone.classifier.in_features
        block = nn.Sequential(
            nn.Linear(self.numfeat, 1024),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.Dropout(0.4),
            nn.Linear(512, self.num_classes))

        self.backbone.classifier = block
        ##########

    def forward(self, x):
      output = self.backbone(x)
      #print(output.logits)
      return output.logits



    def get_mean_std(self):
      return (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)
      #return self.backbone.default_cfg["mean"], self.backbone.default_cfg["std"]


In [ ]:
import gc
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(47)

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
num_class  = 9
lr         = 1e-4
num_epochs  = 100
batch_size = 64
img_size   = 224

model_cnn = GVIT(num_classes=num_class).to(device)
#criterion  = nn.CrossEntropyLoss().to(device)
optimizer =  torch.optim.AdamW(model_cnn.parameters(), lr=lr)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)




In [ ]:
dm = DataModule(model_cnn.get_mean_std(), img_size = img_size, batch_size=batch_size, train_dir = train_dir, valid_dir=valid_dir)
dm.setup()
train_loader = dm.train_dataloader()
valid_loader = dm.val_dataloader()
#test_loader = dm.test_dataloader()

In [ ]:
# =================================================================
# CÁLCULO DE PESOS
# =================================================================
# 1. Extraemos las etiquetas usando tu variable 'dm'
etiquetas_train = dm.traindata.targets

# 2. Contamos cuántas imágenes hay por clase y aplicamos la fórmula
conteo_clases = torch.bincount(torch.tensor(etiquetas_train))
total_muestras = len(etiquetas_train)
num_clases = len(conteo_clases)

pesos_clases = total_muestras / (num_clases * conteo_clases.float())
pesos_tensor = pesos_clases.to(device).float()


print("="*50)
print(f"⚖️ Pesos matemáticos calculados para las {num_clases} clases:")
print(pesos_tensor)
print("="*50)

# 3 CRITERION CON PESOS INYECTADOS
#criterion = nn.CrossEntropyLoss(weight=pesos_tensor, label_smoothing=0.1).to(device)
criterion = nn.CrossEntropyLoss(weight=pesos_tensor).to(device)

In [ ]:
model_cnn, history, lr_history = train(model_cnn, criterion, device, train_loader, valid_loader, optimizer, scheduler, num_epochs, porc = 0.3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Simulación de Datos (Reemplaza 'history' con tus datos reales) ---
# history = [ [loss_train, loss_valid, acc_train, acc_valid], ... ]
arr = np.array(history)
num_epochs = arr.shape[0]
epochs_range = np.arange(0, num_epochs) # Creamos el rango de 0 a N
# ----------------------------------------------------------------------

# Definimos cada cuánto queremos ver los números en el eje X para que no se amontonen
# Si son pocas épocas (menos de 20), mostramos todas. Si son muchas, saltamos.
step = max(1, num_epochs // 10)
mis_ticks = np.arange(0, num_epochs, step)

# ==========================================
# GRÁFICO 1: PÉRDIDA (LOSS)
# ==========================================
plt.figure(figsize=(10, 6))
plt.title('Historial de Pérdida (Loss)', fontsize=16)

plt.plot(epochs_range, arr[:, 0], label='Loss_train', linewidth=2.5)
plt.plot(epochs_range, arr[:, 1], label='Loss_valid', linewidth=2.5, linestyle='--')

plt.ylabel('Pérdida (Loss)', fontsize=12)
plt.xlabel('Época (Epoch)', fontsize=12)

# AQUÍ ASEGURAMOS LOS NUMERITOS EN EL EJE X
plt.xticks(mis_ticks)

plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


# ==========================================
# GRÁFICO 2: PRECISIÓN (ACCURACY) - Sin Acc_valid
# ==========================================
plt.figure(figsize=(10, 6))
plt.title('Historial de Precisión (Accuracy)', fontsize=16)

# Solo ploteamos la columna 2 (Train), ignoramos la 3 (Valid)
plt.plot(epochs_range, arr[:, 2], label='Acc_train', color='green', linewidth=2.5)

plt.ylabel('Precisión (Acc)', fontsize=12)
plt.xlabel('Época (Epoch)', fontsize=12)

# AQUÍ ASEGURAMOS LOS NUMERITOS EN EL EJE X TAMBIÉN
plt.xticks(mis_ticks)

plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### **Entrenando el Modelo** - Cargando el mejor Modelo

In [ ]:
model_sv = GVIT(num_classes=num_class).to(device)
path = "/content/new_dataGVIT/NetModel.pth"
model_sv.load_state_dict(torch.load(path))

In [ ]:
loader = valid_loader
labels = list(dm.validdata.class_to_idx.keys())
y_pred, y_true = test(model_sv, criterion, device, loader, labels)

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
tot = np.sum(cm, axis=1)
cm_porc = np.divide(cm, np.reshape(tot, (-1,1)))

In [ ]:
#@title Matriz de confusión en porcentajes
labels =list(dm.validdata.class_to_idx.keys())
# Calculate confusion matrix
#confusion_matrix = sk_metrics.confusion_matrix(y_true, y_pred)
df_confusion_matrix = pd.DataFrame(cm_porc, index=labels, columns=labels)

# Show confusion matrix
plt.figure(figsize=(6, 6))
sn.heatmap(df_confusion_matrix, annot=True, cbar=False, cmap='Oranges', linewidths=1, linecolor='black', fmt=".2%")
plt.xlabel('Predicted labels', fontsize=15)
plt.xticks(fontsize=12)
plt.ylabel('True labels', fontsize=15)
plt.yticks(fontsize=12);

In [ ]:
report = classification_report(y_true, y_pred, target_names=labels, output_dict=True)
print(classification_report(y_true, y_pred, target_names=labels))

# Swin_transformer

In [ ]:
def saveModel(model):
    dir_Save ="/content/new_dataSWIN"
    os.makedirs(dir_Save, exist_ok=True)
    path = os.path.join(dir_Save, "NetModel.pth")
    torch.save(model.state_dict(), path)

In [ ]:
import torch
import torch.nn as nn
import timm

class CNN_SWIN(nn.Module):
    def __init__(self, num_classes=10, pretrained=True):
        super().__init__()
        self.num_classes = num_classes

        # Initialize with num_classes=0 to get features without a classifier head
        self.swin = timm.create_model("swin_base_patch4_window7_224", pretrained=pretrained, num_classes=10)

        if pretrained:
            for param in self.swin.parameters():
                param.requires_grad = False

        # Dummy forward para obtener el tamaño de las features
        dummy_input = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            # forward_features devuelve (Batch, H, W, C) en Swin
            features = self.swin.forward_features(dummy_input)
            self.numfeat = features.shape[-1]

        self.classifier_head = nn.Sequential(
            nn.Linear(self.numfeat, 1024),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.Dropout(0.4),
            nn.Linear(512, self.num_classes)
        )

    def forward(self, x):
        # 1. Obtener features del backbone
        # Shape output: (Batch, H, W, C) -> Ejemplo: (128, 7, 7, 1024)
        features = self.swin.forward_features(x)

        # 2. APLICAR GLOBAL POOLING (ESTA ES LA CORRECCIÓN)
        # Necesitamos convertir (Batch, H, W, C) -> (Batch, C)
        # Movemos las dimensiones para que coincida con el formato de PyTorch (B, C, H, W)
        features = features.permute(0, 3, 1, 2)
        # Aplicamos un promedio global para reducir H y W a 1x1
        features = nn.functional.adaptive_avg_pool2d(features, (1, 1))
        # Aplanamos para que quede (Batch, Channels)
        features = features.flatten(1)

        # 3. Aplicar el clasificador
        # Input shape: (Batch, 1024) -> Output: (Batch, 10)
        out = self.classifier_head(features)
        return out

    def get_mean_std(self):
        return self.swin.default_cfg["mean"], self.swin.default_cfg["std"]

In [ ]:
import gc
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(47)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
num_class  = 9
lr         = 1e-4
num_epochs  = 100
batch_size = 64
img_size   = 224

model_cnn = CNN_SWIN(num_classes=num_class).to(device)
#criterion  = nn.CrossEntropyLoss().to(device)
optimizer =  torch.optim.AdamW(model_cnn.parameters(), lr=lr)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)




In [ ]:
dm = DataModule(model_cnn.get_mean_std(), img_size = img_size, batch_size=batch_size, train_dir = train_dir, valid_dir=valid_dir)
dm.setup()
train_loader = dm.train_dataloader()
valid_loader = dm.val_dataloader()
#test_loader = dm.test_dataloader()

In [ ]:
# =================================================================
# CÁLCULO DE PESOS
# =================================================================
# 1. Extraemos las etiquetas usando tu variable 'dm'
etiquetas_train = dm.traindata.targets

# 2. Contamos cuántas imágenes hay por clase y aplicamos la fórmula
conteo_clases = torch.bincount(torch.tensor(etiquetas_train))
total_muestras = len(etiquetas_train)
num_clases = len(conteo_clases)

pesos_clases = total_muestras / (num_clases * conteo_clases.float())
pesos_tensor = pesos_clases.to(device).float()


print("="*50)
print(f"⚖️ Pesos matemáticos calculados para las {num_clases} clases:")
print(pesos_tensor)
print("="*50)

# 3 CRITERION CON PESOS INYECTADOS
#criterion = nn.CrossEntropyLoss(weight=pesos_tensor, label_smoothing=0.1).to(device)
criterion = nn.CrossEntropyLoss(weight=pesos_tensor).to(device)

In [ ]:
model_cnn, history, lr_history = train(model_cnn, criterion, device, train_loader, valid_loader, optimizer, scheduler, num_epochs, porc = 0.3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Simulación de Datos (Reemplaza 'history' con tus datos reales) ---
# history = [ [loss_train, loss_valid, acc_train, acc_valid], ... ]
arr = np.array(history)
num_epochs = arr.shape[0]
epochs_range = np.arange(0, num_epochs) # Creamos el rango de 0 a N
# ----------------------------------------------------------------------

# Definimos cada cuánto queremos ver los números en el eje X para que no se amontonen
# Si son pocas épocas (menos de 20), mostramos todas. Si son muchas, saltamos.
step = max(1, num_epochs // 10)
mis_ticks = np.arange(0, num_epochs, step)

# ==========================================
# GRÁFICO 1: PÉRDIDA (LOSS)
# ==========================================
plt.figure(figsize=(10, 6))
plt.title('Historial de Pérdida (Loss)', fontsize=16)

plt.plot(epochs_range, arr[:, 0], label='Loss_train', linewidth=2.5)
plt.plot(epochs_range, arr[:, 1], label='Loss_valid', linewidth=2.5, linestyle='--')

plt.ylabel('Pérdida (Loss)', fontsize=12)
plt.xlabel('Época (Epoch)', fontsize=12)

# AQUÍ ASEGURAMOS LOS NUMERITOS EN EL EJE X
plt.xticks(mis_ticks)

plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


# ==========================================
# GRÁFICO 2: PRECISIÓN (ACCURACY) - Sin Acc_valid
# ==========================================
plt.figure(figsize=(10, 6))
plt.title('Historial de Precisión (Accuracy)', fontsize=16)

# Solo ploteamos la columna 2 (Train), ignoramos la 3 (Valid)
plt.plot(epochs_range, arr[:, 2], label='Acc_train', color='green', linewidth=2.5)

plt.ylabel('Precisión (Acc)', fontsize=12)
plt.xlabel('Época (Epoch)', fontsize=12)

# AQUÍ ASEGURAMOS LOS NUMERITOS EN EL EJE X TAMBIÉN
plt.xticks(mis_ticks)

plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### **Entrenando el Modelo** - Cargando el mejor Modelo

In [ ]:
model_sv = CNN_SWIN(num_classes=num_class).to(device)
path = "/content/new_dataSWIN/NetModel.pth"
model_sv.load_state_dict(torch.load(path))

In [ ]:
loader = valid_loader
labels = list(dm.validdata.class_to_idx.keys())
y_pred, y_true = test(model_sv, criterion, device, loader, labels)

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
tot = np.sum(cm, axis=1)
cm_porc = np.divide(cm, np.reshape(tot, (-1,1)))

In [ ]:
#@title Matriz de confusión en porcentajes
labels =list(dm.validdata.class_to_idx.keys())
# Calculate confusion matrix
#confusion_matrix = sk_metrics.confusion_matrix(y_true, y_pred)
df_confusion_matrix = pd.DataFrame(cm_porc, index=labels, columns=labels)

# Show confusion matrix
plt.figure(figsize=(6, 6))
sn.heatmap(df_confusion_matrix, annot=True, cbar=False, cmap='Oranges', linewidths=1, linecolor='black', fmt=".2%")
plt.xlabel('Predicted labels', fontsize=15)
plt.xticks(fontsize=12)
plt.ylabel('True labels', fontsize=15)
plt.yticks(fontsize=12);

In [ ]:
report = classification_report(y_true, y_pred, target_names=labels, output_dict=True)
print(classification_report(y_true, y_pred, target_names=labels))

# nateraw/vit-base-patch16-224-cifar10

In [ ]:
def saveModel(model):
    dir_Save ="/content/Nat"
    os.makedirs(dir_Save, exist_ok=True)
    path = os.path.join(dir_Save, "NetModel.pth")
    torch.save(model.state_dict(), path)

In [ ]:
from PIL import Image
import requests
from transformers import ViTImageProcessor, ViTForImageClassification

class NaterawVIT(nn.Module):
    def __init__(self, num_classes=10, pretrained=True):
        super().__init__()

        self.num_classes = num_classes

        ########
        # Cargar el procesador de imágenes y el modelo preentrenado
        self.processor = ViTImageProcessor.from_pretrained('nateraw/vit-base-patch16-224-cifar10')
        self.backbone = ViTForImageClassification.from_pretrained('nateraw/vit-base-patch16-224-cifar10')

        if pretrained:
            # Congelar los pesos del modelo
            for param in self.backbone.parameters():
                param.requires_grad = False

        # Obtener el número de características de la capa clasificadora
        self.numfeat = self.backbone.classifier.in_features

        # Definir un nuevo bloque de clasificación
        block = nn.Sequential(
            nn.Linear(self.numfeat, 1024),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.Dropout(0.4),
            nn.Linear(512, self.num_classes)
        )

        # Reemplazar la capa clasificadora del modelo con el nuevo bloque
        self.backbone.classifier = block
        ##########

    def forward(self, x):
        output = self.backbone(x)
        return output.logits

    def get_mean_std(self):
        return (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)


In [ ]:
import gc
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(47)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
num_class  = 9
lr         = 1e-4
num_epochs  = 100
batch_size = 64
img_size   = 224

model_cnn = NaterawVIT(num_classes=num_class).to(device)
#criterion  = nn.CrossEntropyLoss().to(device)
optimizer =  torch.optim.AdamW(model_cnn.parameters(), lr=lr)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)

In [ ]:
dm = DataModule(model_cnn.get_mean_std(), img_size = img_size, batch_size=batch_size, train_dir = train_dir, valid_dir=valid_dir)
dm.setup()
train_loader = dm.train_dataloader()
valid_loader = dm.val_dataloader()
#test_loader = dm.test_dataloader()

In [ ]:
# =================================================================
# CÁLCULO DE PESOS
# =================================================================
# 1. Extraemos las etiquetas usando tu variable 'dm'
etiquetas_train = dm.traindata.targets

# 2. Contamos cuántas imágenes hay por clase y aplicamos la fórmula
conteo_clases = torch.bincount(torch.tensor(etiquetas_train))
total_muestras = len(etiquetas_train)
num_clases = len(conteo_clases)

pesos_clases = total_muestras / (num_clases * conteo_clases.float())
pesos_tensor = pesos_clases.to(device).float()


print("="*50)
print(f"⚖️ Pesos matemáticos calculados para las {num_clases} clases:")
print(pesos_tensor)
print("="*50)

# 3 CRITERION CON PESOS INYECTADOS
#criterion = nn.CrossEntropyLoss(weight=pesos_tensor, label_smoothing=0.1).to(device)
criterion = nn.CrossEntropyLoss(weight=pesos_tensor).to(device)

In [ ]:
model_cnn, history, lr_history = train(model_cnn, criterion, device, train_loader, valid_loader, optimizer, scheduler, num_epochs, porc = 0.3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Simulación de Datos (Reemplaza 'history' con tus datos reales) ---
# history = [ [loss_train, loss_valid, acc_train, acc_valid], ... ]
arr = np.array(history)
num_epochs = arr.shape[0]
epochs_range = np.arange(0, num_epochs) # Creamos el rango de 0 a N
# ----------------------------------------------------------------------

# Definimos cada cuánto queremos ver los números en el eje X para que no se amontonen
# Si son pocas épocas (menos de 20), mostramos todas. Si son muchas, saltamos.
step = max(1, num_epochs // 10)
mis_ticks = np.arange(0, num_epochs, step)

# ==========================================
# GRÁFICO 1: PÉRDIDA (LOSS)
# ==========================================
plt.figure(figsize=(10, 6))
plt.title('Historial de Pérdida (Loss)', fontsize=16)

plt.plot(epochs_range, arr[:, 0], label='Loss_train', linewidth=2.5)
plt.plot(epochs_range, arr[:, 1], label='Loss_valid', linewidth=2.5, linestyle='--')

plt.ylabel('Pérdida (Loss)', fontsize=12)
plt.xlabel('Época (Epoch)', fontsize=12)

# AQUÍ ASEGURAMOS LOS NUMERITOS EN EL EJE X
plt.xticks(mis_ticks)

plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


# ==========================================
# GRÁFICO 2: PRECISIÓN (ACCURACY) - Sin Acc_valid
# ==========================================
plt.figure(figsize=(10, 6))
plt.title('Historial de Precisión (Accuracy)', fontsize=16)

# Solo ploteamos la columna 2 (Train), ignoramos la 3 (Valid)
plt.plot(epochs_range, arr[:, 2], label='Acc_train', color='green', linewidth=2.5)

plt.ylabel('Precisión (Acc)', fontsize=12)
plt.xlabel('Época (Epoch)', fontsize=12)

# AQUÍ ASEGURAMOS LOS NUMERITOS EN EL EJE X TAMBIÉN
plt.xticks(mis_ticks)

plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### **Entrenando el Modelo** - Cargando el mejor Modelo

In [ ]:
model_sv = NaterawVIT(num_classes=num_class).to(device)
path = "/content/Nat/NetModel.pth"
model_sv.load_state_dict(torch.load(path))


In [ ]:
loader = valid_loader
labels = list(dm.validdata.class_to_idx.keys())
y_pred, y_true = test(model_sv, criterion, device, loader, labels)

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
tot = np.sum(cm, axis=1)
cm_porc = np.divide(cm, np.reshape(tot, (-1,1)))


In [ ]:
#@title Matriz de confusión en porcentajes
labels =list(dm.validdata.class_to_idx.keys())
# Calculate confusion matrix
#confusion_matrix = sk_metrics.confusion_matrix(y_true, y_pred)
df_confusion_matrix = pd.DataFrame(cm_porc, index=labels, columns=labels)

# Show confusion matrix
plt.figure(figsize=(6, 6))
sn.heatmap(df_confusion_matrix, annot=True, cbar=False, cmap='Oranges', linewidths=1, linecolor='black', fmt=".2%")
plt.xlabel('Predicted labels', fontsize=15)
plt.xticks(fontsize=12)
plt.ylabel('True labels', fontsize=15)
plt.yticks(fontsize=12);

In [ ]:
report = classification_report(y_true, y_pred, target_names=labels, output_dict=True)
print(classification_report(y_true, y_pred, target_names=labels))



# Microsoft-VIT-beit-patch16-224-pt22k-ft22k



In [ ]:
def saveModel(model):
    dir_Save ="/content/new_Microsoft"
    os.makedirs(dir_Save, exist_ok=True)
    path = os.path.join(dir_Save, "NetModel.pth")
    torch.save(model.state_dict(), path)

In [ ]:
import gc
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(48110)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import torch
import torch.nn as nn
from transformers import BeitImageProcessor
from transformers.models.beit.modeling_beit import BeitForImageClassification

class MicrosoftBEiT(nn.Module):
    def __init__(self, num_classes=9, pretrained=True):
        super().__init__()

        self.num_classes = num_classes

        ########
        # Cargar el procesador de imágenes y el modelo preentrenado
        self.processor = BeitImageProcessor.from_pretrained('microsoft/beit-base-patch16-224-pt22k-ft22k')
        self.backbone = BeitForImageClassification.from_pretrained('microsoft/beit-base-patch16-224-pt22k-ft22k')

        if pretrained:
            # Congelar los pesos del modelo
            for param in self.backbone.parameters():
                param.requires_grad = False

        # Obtener el número de características de la capa clasificadora
        self.numfeat = self.backbone.classifier.in_features

        # Definir un nuevo bloque de clasificación
        block = nn.Sequential(
            nn.Linear(self.numfeat, 1024),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.Dropout(0.4),
            nn.Linear(512, self.num_classes)
        )

        # Reemplazar la capa clasificadora del modelo con el nuevo bloque
        self.backbone.classifier = block
        ##########

    def forward(self, x):
        output = self.backbone(x)
        return output.logits

    def get_mean_std(self):
        return (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)


In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
num_class  = 9
lr         = 1e-4
num_epochs  = 100
batch_size = 64
img_size   = 224

model_cnn = MicrosoftBEiT(num_classes=num_class, pretrained=True).to(device)
#criterion  = nn.CrossEntropyLoss().to(device)
optimizer =  torch.optim.AdamW(model_cnn.parameters(), lr=lr)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)

In [ ]:
dm = DataModule(model_cnn.get_mean_std(), img_size = img_size, batch_size=batch_size, train_dir = train_dir, valid_dir=valid_dir)
dm.setup()
train_loader = dm.train_dataloader()
valid_loader = dm.val_dataloader()
#test_loader = dm.test_dataloader()

In [ ]:
# =================================================================
# CÁLCULO DE PESOS
# =================================================================
# 1. Extraemos las etiquetas usando tu variable 'dm'
etiquetas_train = dm.traindata.targets

# 2. Contamos cuántas imágenes hay por clase y aplicamos la fórmula
conteo_clases = torch.bincount(torch.tensor(etiquetas_train))
total_muestras = len(etiquetas_train)
num_clases = len(conteo_clases)

pesos_clases = total_muestras / (num_clases * conteo_clases.float())
pesos_tensor = pesos_clases.to(device).float()


print("="*50)
print(f"⚖️ Pesos matemáticos calculados para las {num_clases} clases:")
print(pesos_tensor)
print("="*50)

# 3 CRITERION CON PESOS INYECTADOS
#criterion = nn.CrossEntropyLoss(weight=pesos_tensor, label_smoothing=0.1).to(device)
criterion = nn.CrossEntropyLoss(weight=pesos_tensor).to(device)

In [ ]:
# Suponiendo que tu variable se llama 'train_dataset'
print(dm.validdata.class_to_idx.keys())


In [ ]:
# Entrenamos el modelo
model, history, lr_history = train(model_cnn, criterion, device, train_loader, valid_loader, optimizer, scheduler, num_epochs, porc=0.3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Simulación de Datos (Reemplaza 'history' con tus datos reales) ---
# history = [ [loss_train, loss_valid, acc_train, acc_valid], ... ]
arr = np.array(history)
num_epochs = arr.shape[0]
epochs_range = np.arange(0, num_epochs) # Creamos el rango de 0 a N
# ----------------------------------------------------------------------

# Definimos cada cuánto queremos ver los números en el eje X para que no se amontonen
# Si son pocas épocas (menos de 20), mostramos todas. Si son muchas, saltamos.
step = max(1, num_epochs // 10)
mis_ticks = np.arange(0, num_epochs, step)

# ==========================================
# GRÁFICO 1: PÉRDIDA (LOSS)
# ==========================================
plt.figure(figsize=(10, 6))
plt.title('Historial de Pérdida (Loss)', fontsize=16)

plt.plot(epochs_range, arr[:, 0], label='Loss_train', linewidth=2.5)
plt.plot(epochs_range, arr[:, 1], label='Loss_valid', linewidth=2.5, linestyle='--')

plt.ylabel('Pérdida (Loss)', fontsize=12)
plt.xlabel('Época (Epoch)', fontsize=12)

# AQUÍ ASEGURAMOS LOS NUMERITOS EN EL EJE X
plt.xticks(mis_ticks)

plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


# ==========================================
# GRÁFICO 2: PRECISIÓN (ACCURACY) - Sin Acc_valid
# ==========================================
plt.figure(figsize=(10, 6))
plt.title('Historial de Precisión (Accuracy)', fontsize=16)

# Solo ploteamos la columna 2 (Train), ignoramos la 3 (Valid)
plt.plot(epochs_range, arr[:, 2], label='Acc_train', color='green', linewidth=2.5)

plt.ylabel('Precisión (Acc)', fontsize=12)
plt.xlabel('Época (Epoch)', fontsize=12)

# AQUÍ ASEGURAMOS LOS NUMERITOS EN EL EJE X TAMBIÉN
plt.xticks(mis_ticks)

plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### **Entrenando el Modelo** - Cargando el mejor Modelo

In [ ]:
model_sv = MicrosoftBEiT(num_classes=num_class).to(device)
path = "/content/new_Microsoft/NetModel.pth"
model_sv.load_state_dict(torch.load(path))

In [ ]:
loader = valid_loader
labels = list(dm.validdata.class_to_idx.keys())
y_pred, y_true = test(model_sv, criterion, device, loader, labels)

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
tot = np.sum(cm, axis=1)
cm_porc = np.divide(cm, np.reshape(tot, (-1,1)))

In [ ]:
#@title Matriz de confusión en porcentajes
labels =list(dm.validdata.class_to_idx.keys())
# Calculate confusion matrix
#confusion_matrix = sk_metrics.confusion_matrix(y_true, y_pred)
df_confusion_matrix = pd.DataFrame(cm_porc, index=labels, columns=labels)

# Show confusion matrix
plt.figure(figsize=(6, 6))
sn.heatmap(df_confusion_matrix, annot=True, cbar=False, cmap='Oranges', linewidths=1, linecolor='black', fmt=".2%")
plt.xlabel('Predicted labels', fontsize=15)
plt.xticks(fontsize=12)
plt.ylabel('True labels', fontsize=15)
plt.yticks(fontsize=12);

In [ ]:
report = classification_report(y_true, y_pred, target_names=labels, output_dict=True)
print(classification_report(y_true, y_pred, target_names=labels))


# MambaVision-L-21K



In [ ]:
def saveModel(model):
    dir_Save ="/content/new_MAMBA"
    os.makedirs(dir_Save, exist_ok=True)
    path = os.path.join(dir_Save, "NetModel.pth")
    torch.save(model.state_dict(), path)

In [ ]:
from transformers import AutoModel
import torch
import torch.nn as nn
from transformers import AutoModelForImageClassification
from timm.data.transforms_factory import create_transform

class MambaVisionCustom(nn.Module):
    def __init__(self, num_classes=10, pretrained=True):
        super().__init__()
        self.num_classes = num_classes

        # 1. Cargar el modelo
        self.backbone = AutoModelForImageClassification.from_pretrained(
            "nvidia/MambaVision-L-21K",
            trust_remote_code=True
        )

        # 2. Congelar pesos
        if pretrained:
            for param in self.backbone.parameters():
                param.requires_grad = False

        # 3. Reemplazar el clasificador de forma directa
        # En tu versión, backbone.model.head es el objeto Linear directamente
        if hasattr(self.backbone, 'model') and hasattr(self.backbone.model, 'head'):
            # Sacamos los rasgos de entrada directamente del Linear
            self.numfeat = self.backbone.model.head.in_features
            # Reemplazamos el Linear por tu Sequential personalizado
            self.backbone.model.head = self._build_custom_head()
        else:
            # Plan de respaldo por si la jerarquía cambia
            self.numfeat = self.backbone.classifier.in_features
            self.backbone.classifier = self._build_custom_head()

    def _build_custom_head(self):
        return nn.Sequential(
            nn.Linear(self.numfeat, 1024),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.Dropout(0.4),
            nn.Linear(512, self.num_classes)
        )

    def forward(self, x):
        # 1. Obtenemos la salida del modelo de Hugging Face
        outputs = self.backbone(x)

        # 2. Si la salida es un diccionario o un objeto de HF, extraemos los logits
        if isinstance(outputs, dict) and 'logits' in outputs:
            return outputs['logits']
        elif hasattr(outputs, 'logits'):
            return outputs.logits

        # 3. Si por alguna razón ya es un Tensor, lo devolvemos tal cual
        return outputs

    def get_mean_std(self):
        cfg = self.backbone.config
        mean = getattr(cfg, 'mean', (0.485, 0.456, 0.406))
        std = getattr(cfg, 'std', (0.229, 0.224, 0.225))
        return mean, std

In [ ]:
import gc
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(47)

gc.collect()
torch.cuda.empty_cache()

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
num_class  = 10
lr         = 1e-4
num_epochs  = 50
batch_size = 32
img_size   = 224

model_cnn = MambaVisionCustom(num_classes=num_class, pretrained=True).to(device)
criterion  = nn.CrossEntropyLoss().to(device)
optimizer =  torch.optim.AdamW(model_cnn.parameters(), lr=lr, weight_decay=1e-3)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=3)

In [ ]:
dm = DataModule(model_cnn.get_mean_std(), img_size = img_size, batch_size=batch_size, train_dir = train_dir, valid_dir=valid_dir, test_dir=test_dir )
dm.setup()
train_loader = dm.train_dataloader()
valid_loader = dm.val_dataloader()
test_loader = dm.test_dataloader()

In [ ]:
# Entrenamos el modelo
model, history, lr_history = train(model_cnn, criterion, device, train_loader, valid_loader, optimizer, scheduler, num_epochs, porc=0.3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Simulación de Datos (Reemplaza 'history' con tus datos reales) ---
# history = [ [loss_train, loss_valid, acc_train, acc_valid], ... ]
arr = np.array(history)
num_epochs = arr.shape[0]
epochs_range = np.arange(0, num_epochs) # Creamos el rango de 0 a N
# ----------------------------------------------------------------------

# Definimos cada cuánto queremos ver los números en el eje X para que no se amontonen
# Si son pocas épocas (menos de 20), mostramos todas. Si son muchas, saltamos.
step = max(1, num_epochs // 10)
mis_ticks = np.arange(0, num_epochs, step)

# ==========================================
# GRÁFICO 1: PÉRDIDA (LOSS)
# ==========================================
plt.figure(figsize=(10, 6))
plt.title('Historial de Pérdida (Loss)', fontsize=16)

plt.plot(epochs_range, arr[:, 0], label='Loss_train', linewidth=2.5)
plt.plot(epochs_range, arr[:, 1], label='Loss_valid', linewidth=2.5, linestyle='--')

plt.ylabel('Pérdida (Loss)', fontsize=12)
plt.xlabel('Época (Epoch)', fontsize=12)

# AQUÍ ASEGURAMOS LOS NUMERITOS EN EL EJE X
plt.xticks(mis_ticks)

plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


# ==========================================
# GRÁFICO 2: PRECISIÓN (ACCURACY) - Sin Acc_valid
# ==========================================
plt.figure(figsize=(10, 6))
plt.title('Historial de Precisión (Accuracy)', fontsize=16)

# Solo ploteamos la columna 2 (Train), ignoramos la 3 (Valid)
plt.plot(epochs_range, arr[:, 2], label='Acc_train', color='green', linewidth=2.5)

plt.ylabel('Precisión (Acc)', fontsize=12)
plt.xlabel('Época (Epoch)', fontsize=12)

# AQUÍ ASEGURAMOS LOS NUMERITOS EN EL EJE X TAMBIÉN
plt.xticks(mis_ticks)

plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### **Entrenando el Modelo** - Cargando el mejor Modelo

In [ ]:
model_sv = MambaVisionCustom(num_classes=num_class).to(device)
path = "/content/new_MAMBA/NetModel.pth"
model_sv.load_state_dict(torch.load(path))

In [ ]:
loader = test_loader
labels = list(dm.testdata.class_to_idx.keys())
y_pred, y_true = test(model_sv, criterion, device, loader, labels)

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
tot = np.sum(cm, axis=1)
cm_porc = np.divide(cm, np.reshape(tot, (-1,1)))

In [ ]:
#@title Matriz de confusión en porcentajes
labels =list(dm.testdata.class_to_idx.keys())
# Calculate confusion matrix
#confusion_matrix = sk_metrics.confusion_matrix(y_true, y_pred)
df_confusion_matrix = pd.DataFrame(cm_porc, index=labels, columns=labels)

# Show confusion matrix
plt.figure(figsize=(6, 6))
sn.heatmap(df_confusion_matrix, annot=True, cbar=False, cmap='Oranges', linewidths=1, linecolor='black', fmt=".2%")
plt.xlabel('Predicted labels', fontsize=15)
plt.xticks(fontsize=12)
plt.ylabel('True labels', fontsize=15)
plt.yticks(fontsize=12);

In [ ]:
report = classification_report(y_true, y_pred, target_names=labels, output_dict=True)
print(classification_report(y_true, y_pred, target_names=labels))